# 3DGS Pipeline 控制中枢

**使用方法**：
1. 运行 **Cell 1（初始化）**，加载所有函数和配置
2. 按需运行各 Section 中的单元格
3. 修改参数？编辑 ，重新运行 Cell 1 即可

> 每个 Section 均可独立运行，无需依赖上方单元格的执行状态。

In [2]:
# ═══════════════════════════════════════════════════════
# ① 初始化（每次打开 Notebook 只需运行这一个 Cell）
# ═══════════════════════════════════════════════════════
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
from src.pipeline.saga import *
from pathlib import Path


os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 强制用 RTX 2080 Ti

# 确保 src/ 在 Python 路径中（支持从任意目录打开 notebook）
_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "src" / "pipeline" / "__init__.py").exists()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.pipeline import *

cfg = load_config()   # 读取 configs/pipeline.yaml
print(f"✓ 项目根目录 : {PROJECT_ROOT}")
print(f"✓ 数据集     : {cfg['dataset']['source']}  →  {cfg['dataset']['path']}")
print(f"✓ 训练输出   : {cfg['training']['output_dir']}")
print(f"✓ 迭代次数   : {cfg['training']['iterations']}")
print(f"✓ 查看器     : {cfg['viewer']['backend']}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'src'

## Section 1 — 环境检查

In [2]:
# 检查 PyTorch、CUDA、COLMAP、3DGS CUDA 模块、Open3D、Docker
check_environment(cfg)


3DGS 环境检查

PyTorch:  2.1.2
CUDA 可用: True
CUDA 版本: 11.8
  GPU 0: NVIDIA GeForce RTX 4060  (7.8 GB)

核心依赖:
  ✓ OpenCV  4.13.0
  ✓ NumPy  2.2.6
  ✓ plyfile
  ✓ SciPy  1.15.3
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-04-01 15:39:21,972  INFO  环境检查完成，结果: 通过


  ✓ Open3D  0.19.0
  ✓ diff_gaussian_rasterization (CUDA 模块)

COLMAP:
  ✓ /usr/bin/colmap

Docker（SIBR 查看器）:
  ✓ Docker daemon 可访问

项目目录:
  PROJECT_ROOT : /home/ansatz/github/ME6402-3D-Autonomous-Retail
  GS_DIR       : /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/gaussian-splatting  ✓
  DATA_DIR     : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data  ✓
  OUTPUT_DIR   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs  ✓

✅ 环境检查通过


True

## Section 2 — 视频抽帧（可选）

适用场景：你有一段录制的视频，需要先抽帧再走 COLMAP 流程。

**配置方式**：在  中修改  块，将 ，
然后重新运行 **Cell 1**，再运行本 Cell。

In [ ]:
# 抽帧完成后会自动更新 cfg，指向新场景目录
# 完成后直接运行 Section 3 (COLMAP) 即可
extract_frames(cfg)


## Section 3 — COLMAP 相机标定

适用场景：自有数据（视频抽帧或自拍照片），需要从图像推导相机参数。
官方数据集（T&T、DB）已自带相机参数，**无需此步骤**。

**配置方式**：在  中将 ，
并确认  和  正确。

In [ ]:
# COLMAP 五步流程：特征提取 → 匹配 → 稀疏重建 → 畸变校正 → 内参回填
# 完成后 cfg["dataset"]["path"] 自动切换到 undistorted dense 输出
run_colmap(cfg)


## Section 4 — 3DGS 训练

关键参数（在  →  块修改）：
- ：迭代次数（300~5000 快速验证；30000 高质量）
- ：分辨率倍率（1=原始；2=1/2；RTX 4060 建议从 2 开始）
- ：输出根目录（每次训练自动创建子目录）

In [ ]:
# OOM 时自动降档重试（resolution ×1 → ×2 → ×4）
run_training(cfg)


## Section 5 — 查看训练结果

自动搜索  下最新的 ，用 Open3D 打开交互窗口。

如需查看 SIBR，在  中将 ，
重新运行 Cell 1 后再运行此 Section。

In [ ]:
# Open3D 交互查看（关闭窗口后继续）
# 也可传入指定路径：open_viewer(cfg, ply_path="outputs/xxx/point_cloud/iteration_300/point_cloud.ply")
open_viewer(cfg)


✗ 未找到 point_cloud.ply，请先完成训练（Section 4）
   搜索目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/playroom_5000iter


False

In [ ]:
# 分析训练结果：列出 PLY 文件、打印 results.json
analyze_results(cfg)


## Section 6 — SIBR Viewer（可选）

如需在 SIBR 中查看，运行此 Cell。会列出可用模型供你选择。

**前提**：已构建 Docker 镜像（见 [info] Building image: sibr-builder:ubuntu22.04-cuda11.8
Sending build context to Docker daemon  12.01GB

Step 1/4 : FROM nvidia/cuda:11.8.0-devel-ubuntu22.04
 ---> 6f9cc9f1ba9e
Step 2/4 : ENV DEBIAN_FRONTEND=noninteractive
 ---> Using cache
 ---> db877d8112e1
Step 3/4 : RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     cmake     ninja-build     git     pkg-config     libglew-dev     libassimp-dev     libboost-all-dev     libgtk-3-dev     libopencv-dev     libglfw3-dev     libavdevice-dev     libavcodec-dev     libavformat-dev     libswscale-dev     libeigen3-dev     libxxf86vm-dev     libembree-dev     libgl1-mesa-dev     libglu1-mesa-dev     libx11-dev     libxext-dev     libxrender-dev     libxrandr-dev     libxinerama-dev     libxcursor-dev     ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> feb68a7c28aa
Step 4/4 : WORKDIR /workspace
 ---> Using cache
 ---> 8fdbd6c2ecd5
Successfully built 8fdbd6c2ecd5
Successfully tagged sibr-builder:ubuntu22.04-cuda11.8
[done] Image built: sibr-builder:ubuntu22.04-cuda11.8）

In [8]:
# 交互式选择模型并在 Docker 内启动 SIBR
launch_sibr(cfg)



可用模型（按最新迭代降序）:
  [1] 7dfdb283-b  (iter=30000)
  [2] 3dgs_tandt_30000iter  (iter=30000)
  [3] 3dgs_tandt_5000iter  (iter=5000)
  [4] 3dgs_custom_scene_01_5000iter  (iter=5000)
  [5] 3dgs_demo_300iter  (iter=300)
  [6] smoke_truck_10iter  (iter=10)
  [7] 3dgs_demo  (iter=?)

🖼️  启动 SIBR Viewer ...
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_sibr_in_docker.sh sibr-builder:ubuntu22.04-cuda11.8 /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/Teamate_output_unzipped/content/gaussian-splatting/output/7dfdb283-b
   提示：关闭 SIBR 窗口后，该单元继续运行。
[info] Launching SIBR viewer in container...

== CUDA ==

CUDA Version 11.8.0

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning

[SIBR] ##  ERROR  ##:	FILE /workspace/third_party/gaussian-splatting/SIBR_viewers/src/core/scene/ParseData.cpp
			LINE 560, FUNC getParsedData
			Cannot determine type of dataset at //content/gaussian-splatting/data


[SIBR] --  INFOS  --:	Initializing Raycaster
[SIBR] --  INFOS  --:	Interactive camera using (0.009,1100) near/far planes.
Switched to trackball mode.


2026-04-01 15:14:30,006  INFO  SIBR 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/Teamate_output_unzipped/content/gaussian-splatting/output/7dfdb283-b，ok=True


[SIBR] --  INFOS  --:	Deinitialization of GLFW
[done] SIBR viewer exited.
✓ SIBR 正常退出


True

## Section 7 — 一键完整流程

参数调好后，一口气执行：COLMAP（可选）→ 训练 → 查看结果。

In [ ]:
# 一键流程（是否跑 COLMAP 取决于 cfg["dataset"]["use_colmap"]）
run_pipeline(cfg)


## Section 9 — SAGA 3D 语义分割

**Segment Any 3D Gaussians**（AAAI 2025）：在已训练好的 3DGS 模型上直接添加语义特征，无需重新训练重建，无需标注数据。

**流程概览**
1. 安装 SAGA 依赖（只需做一次）
2. 下载 SAM ViT-H checkpoint（~2.5 GB，只需做一次）
3. 提取 SAM 特征 + mask（按场景做一次）
4. 训练对比特征（~10-40 分钟）
5. 打开 SAGA Notebook → 文字/点击 → 输出 3D Bounding Box

**配置方式**：在 `configs/pipeline.yaml` 的 `saga:` 块中设置 `image_root` 和 `model_path`，然后重新运行 **Cell 1**。

In [20]:
# 9.1  导入 SAGA 模块 + 加载配置
from src.pipeline.saga import *

saga_cfg = load_saga_config(cfg)   # 读取 pipeline.yaml 中的 saga: 块

print(f"模型路径  : {saga_cfg['model_path']}")
print(f"场景图像  : {saga_cfg['image_root']}")
print(f"SAM ckpt  : {saga_cfg['sam_checkpoint']}")
print(f"降采样    : ×{saga_cfg['downsample']}")


模型路径  : outputs/3dgs_custom_scene_01_5000iter
场景图像  : data/colmap_workspace/custom_scene_01/dense
SAM ckpt  : dependencies/sam_ckpt/sam_vit_h_4b8939.pth
降采样    : ×4


In [21]:
# 9.2  SAGA 环境检查（确认依赖全部就绪）
check_saga_ready()


SAGA 环境检查
  ✓ SAGA 仓库  (/home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA)
  ✓ 子模块 diff-gaussian-rasterization_contrastive_f
  ✓ segment-anything (SAM)
  ✓ kmeans_pytorch
  ✓ open_clip_torch
  ✓ hdbscan
  ✓ SAM ViT-H checkpoint  (sam_vit_h_4b8939.pth)

✅ SAGA 环境就绪


True

In [9]:
# 9.3  首次使用：安装 SAGA 依赖
# 若 9.2 显示全部 ✓，可跳过本单元格
import subprocess, sys
result = subprocess.run(
    ["bash", "scripts/setup_saga.sh"],
    cwd=str(PROJECT_ROOT)
)
print("✅ 安装完成" if result.returncode == 0 else "✗ 安装失败，查看上方输出")


 SAGA 安装脚本
 项目根目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ Conda 环境：gaussian_splatting
✓ SAGA 仓库已存在，更新子模块...

>>> 安装 segment-anything...
✓ segment-anything 安装完成（本地源）

>>> 安装 kmeans_pytorch...
✓ kmeans_pytorch 安装完成（本地源）

>>> 安装 open_clip_torch, hdbscan...
✓ open_clip_torch, hdbscan 安装完成

>>> 编译 diff-gaussian-rasterization_contrastive_f...
✓ diff-gaussian-rasterization_contrastive_f 编译完成

>>> 编译 diff-gaussian-rasterization-depth...
✓ diff-gaussian-rasterization-depth 编译完成

 SAGA 依赖安装完成

✓ SAM checkpoint 已存在：/home/ansatz/github/ME6402-3D-Autonomous-Retail/dependencies/sam_ckpt/sam_vit_h_4b8939.pth

下一步：在 Notebook 中打开 SAGA Section 并按顺序运行。
✅ 安装完成


In [6]:
# 9.4  下载 SAM ViT-H checkpoint（~2.5 GB，只需一次）
# 若 dependencies/sam_ckpt/sam_vit_h_4b8939.pth 已存在可跳过
download_sam_checkpoint()


⬇️  正在下载 SAM ViT-H checkpoint → /home/ansatz/github/ME6402-3D-Autonomous-Retail/dependencies/sam_ckpt/sam_vit_h_4b8939.pth
   URL：https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
   大小：约 2.5 GB，请耐心等待...




     0K .......... .......... .......... .......... ..........  0% 3.74M 10m54s
    50K .......... .......... .......... .......... ..........  0% 3.55M 11m11s
   100K .......... .......... .......... .......... ..........  0% 6.06M 9m42s
   150K .......... .......... .......... .......... ..........  0% 6.35M 8m53s
   200K .......... .......... .......... .......... ..........  0%  166M 7m9s
   250K .......... .......... .......... .......... ..........  0% 12.2M 6m31s
   300K .......... .......... .......... .......... ..........  0% 48.7M 5m42s
   350K .......... .......... .......... .......... ..........  0% 3.51M 6m27s
   400K .......... .......... .......... .......... ..........  0% 4.00M 6m52s
   450K .......... .......... .......... .......... ..........  0%  186M 6m12s
   500K .......... .......... .......... .......... ..........  0% 9.10M 6m2s
   550K .......... .......... .......... .......... ..........  0% 15.9M 5m45s
   600K .......... .......... .......... ..........


✓ 下载完成（2446 MB）：/home/ansatz/github/ME6402-3D-Autonomous-Retail/dependencies/sam_ckpt/sam_vit_h_4b8939.pth


........ .......... .......... .......... 99% 6.42M 0s
2502450K .......... .......... .......... .......... .......... 99% 8.22M 0s
2502500K .......... .......... .......... .......... .......... 99%  170M 0s
2502550K .......... .......... .......... .......... .......... 99% 9.87M 0s
2502600K .......... .......... .......... .......... .......... 99%  164M 0s
2502650K .......... .......... .......... .......... .......... 99% 32.8M 0s
2502700K .......... .......... .......... .......... .......... 99% 4.30M 0s
2502750K .......... .......... .......... .......... .......... 99% 37.6M 0s
2502800K .......... .......... .......... .......... .......... 99% 21.8M 0s
2502850K .......... .......... .......... .......... .......... 99% 4.50M 0s
2502900K .......... .......... .......... .......... .......... 99%  217M 0s
2502950K .......... .......... .......... .......... .......... 99% 3.37M 0s
2503000K .......... .......... .......... .......... .......... 99% 2.95M 0s
2503050K .......... .

True

### 9.5 — 指定要分割的场景和模型

修改下方两个变量（或直接在 `configs/pipeline.yaml` 的 `saga:` 块中修改后重新运行 Cell 1）：

```
saga:
  image_root: data/official/tandt_db/db/playroom   # 场景图像目录（含 images/ 子目录）
  model_path: outputs/playroom_30000iter            # 训练好的 3DGS 模型目录
```

> ⚠️ Teammate 的模型（`Teamate_output_unzipped/...`）训练于 Google Colab，本地没有对应图像，需使用本地训练的模型或用自己拍摄的数据。

In [13]:
# 9.5  在这里覆盖场景和模型路径（不影响主 cfg）
saga_cfg["image_root"] = str(PROJECT_ROOT / "data/official/tandt_db/db/playroom")
saga_cfg["model_path"] = str(PROJECT_ROOT / "outputs/3dgs_tandt_30000iter")

# 确认路径
from pathlib import Path
print(f"场景图像目录 : {saga_cfg['image_root']}")
print(f"  images/ 存在: {(Path(saga_cfg['image_root']) / 'images').exists()}")
print(f"模型目录     : {saga_cfg['model_path']}")
print(f"  cfg_args 存在: {(Path(saga_cfg['model_path']) / 'cfg_args').exists()}")


场景图像目录 : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom
  images/ 存在: True
模型目录     : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_tandt_30000iter
  cfg_args 存在: True


### 9.6 — 提取 SAM 特征 + 训练（按顺序运行一次即可）

每个场景只需运行一次，结果会缓存在场景目录下。

In [12]:
# 9.6.1  Step 1：提取 SAM 图像编码特征（输出到 <image_root>/features/）
# VRAM：~7 GB，建议在 RTX 2080 Ti 上运行
extract_sam_features(saga_cfg)


   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom/images（225 张）

[SAGA Step 1] 提取 CLIP 特征（从 SAM mask）...
   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom/images
   输出目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom/clip_features
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/get_clip_features.py --image_root ...
Embedding dimension 512

0it [00:00, ?it/s]/home/ansatz/miniconda3/envs/gaussian_splatting/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antia

True

In [25]:
# 9.6.2  Step 2&3：生成缩小图像 + 提取 SAM 自动分割 mask（输出到 <image_root>/sam_masks/）
# VRAM：~7 GB，建议在 RTX 2080 Ti 上运行
extract_sam_masks(saga_cfg)


✓ images_4/ 已存在，跳过创建。

[SAGA Step 3] 提取 SAM 自动分割 mask...
   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/custom_scene_01/dense/images_4
   输出目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/custom_scene_01/dense/sam_masks
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/extract_segment_everything_masks.py --image_root ...
Initializing SAM...
Extracting SAM segment everything masks...

  0%|          | 0/99 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/extract_segment_everything_masks.py", line 61, in <module>
    masks = mask_generator.generate(img)
  File "/home/ansatz/miniconda3/envs/gaussian_splatting/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
    return func(*args, **kwargs)
  File "/home/ansatz/github/ME6402-3D-Autonomous-Reta

False

In [9]:
# 9.6.3  Step 4：在冻结的 3DGS 上训练对比特征（~10-40 分钟）
# 输出：<model_path>/point_cloud/.../contrastive_feature_point_cloud.ply
train_saga_features(saga_cfg)


[SAGA Step 4] 训练对比特征...
   模型路径：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_tandt_30000iter
   场景数据：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom
   预计时间：10~40 分钟
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/train_contrastive_feature.py -m ...
Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_tandt_30000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_tandt_30000iter/cfg_args
Optimizing /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_tandt_30000iter
RFN weight: 1.0 [01/04 15:54:24]
Smooth K: 16 [01/04 15:54:24]
Scale aware dim: -1 [01/04 15:54:24]
Loading trained model at iteration None, None [01/04 15:54:24]
Allow Camera Principle Point Shift: True [01/04 15:54:24]

Reading camera 1/225Traceback (most recent call last):
  File "/home/ansatz/github/ME640

False

In [ ]:
# 9.6.4  Step 5（可选）：估算 mask 3D 物理尺度，提升分割精度
compute_scales(saga_cfg)


### 9.7 — 交互式 3D 分割

训练完成后，有两种方式进行分割：

**方式 A（推荐）**：打开 SAGA 原生 Notebook，支持文字查询 + 点击 prompt  
**方式 B**：从已保存的 mask 文件直接读取 3D Bounding Box

In [ ]:
# 9.7.1  方式 A：启动 SAGA 交互 Notebook（在浏览器中打开）
# 在 Notebook 中输入文字（如 "milk carton"）或点击物体 → 输出 segmentation_res/final_mask.pt
open_saga_notebook(saga_cfg)


In [ ]:
# 9.7.2  方式 B：从 SAGA Notebook 生成的 mask 读取 3D Bounding Box
# 运行完 SAGA Notebook 后，mask 保存在 third_party/SAGA/segmentation_res/final_mask.pt

mask_file = PROJECT_ROOT / "third_party/SAGA/segmentation_res/final_mask.pt"

bbox = get_3d_bbox_from_mask(
    mask_pt   = mask_file,
    model_path= saga_cfg["model_path"],
    label     = "yogurt bottle",   # 修改为你查询的物体名称
    padding   = 0.05,
)


In [ ]:
# 9.7.3  批量导出多个物体的 3D Bounding Box 为 JSON
# 每分割一个物体后调用一次 get_3d_bbox_from_mask()，收集结果后统一导出

# 示例：假设已分割了 3 个物体
all_bboxes = [
    # get_3d_bbox_from_mask(mask_file, saga_cfg["model_path"], label="product_A"),
    # get_3d_bbox_from_mask(mask_file, saga_cfg["model_path"], label="product_B"),
]

if all_bboxes:
    export_bboxes(all_bboxes)   # 默认输出到 outputs/saga_bboxes.json
else:
    print("ℹ️  先在 SAGA Notebook 中完成分割，再运行本单元格")


## Section 8 — 日志

In [15]:
import subprocess
from src.pipeline import LOG_DIR

logs = sorted(LOG_DIR.glob("pipeline_*.log"), reverse=True)
if logs:
    latest = logs[0]
    print(f"最新日志：{latest}
" + "-"*50)
    # 打印最后 30 行
    lines = latest.read_text(encoding="utf-8").splitlines()
    print("
".join(lines[-30:]))
else:
    print(f"日志目录：{LOG_DIR}（暂无日志）")


SyntaxError: unterminated string literal (detected at line 7) (3844513438.py, line 7)